# BODAQS Data Explorer

This notebook consumes artifacts produced by the batch pre-processing pipeline and displays the data using generic widgets.

 - Metric histogram widget
 - Event browser widget
 - Metric scatter widget
 - Signal histogram widget
 - Session browser widget

**Important: Run all the cells in this notebook**
The final cell sets up the hooks that update the widgets if a new session is selected. Once all the cells have been run, explore at your leisure.

### First code cell: set or edit the directory path to the library

Running this cell allows the directory path to the run library to be specified. 


In [1]:
from pathlib import Path
from IPython.display import display
from bodaqs_analysis.ui import make_preprocess_runtime_settings_editor

runtime_settings_editor = make_preprocess_runtime_settings_editor(
    artifacts_dir=Path("artifacts/Neil"),
    bike_profile_path=None,
    fit_dir=None,
    fit_bindings_path=None,
    prompt_for_descriptions=False,
    show_preprocess_profile_path=False,
    show_generic_log_metadata=False,
    show_logger_timezone=False,
    show_bike_profile_path=False,
    show_fit_inputs=False,
    show_prompt_for_descriptions=False,
    show_run_tz_label=False,
)
display(runtime_settings_editor.ui)


### Second code cell: run selector

Select one or more sessions or aggregations to be charted. If multiple selections exist, they can be compared using the data visualisation widgets. 

In [4]:
from pathlib import Path
from bodaqs_analysis.widgets.session_selector import make_session_selector
from bodaqs_analysis.schema import load_event_schema
from bodaqs_analysis.artifacts import load_session_artifacts
from bodaqs_analysis.widgets.loaders import make_session_loader
    
runtime_settings = runtime_settings_editor.get_settings()
ARTIFACTS_DIR = runtime_settings["artifacts_dir"]

SCHEMA_PATH = Path(r"event schema\event_schema.yaml")
schema = load_event_schema(SCHEMA_PATH)

sel = make_session_selector(artifacts_dir=ARTIFACTS_DIR, select_first_by_default=True)
display(sel["ui"])

events_index_df = sel["get_events_index_df"]()
key_to_ref = sel["get_key_to_ref"]()
store = sel["store"]

session_loader = make_session_loader(store=store, key_to_ref=key_to_ref)


In [6]:
TARGET_STUDY_SET_ID = "test-study-set-1"  # e.g. "setup-comparison-1"

if TARGET_STUDY_SET_ID.strip():
    study_set_bridge = adapter.study_set_to_selection_snapshot(
        LIBRARY_ID,
        TARGET_STUDY_SET_ID.strip(),
        include_groupings=False,
    )
    study_set_selector_handle = study_set_bridge["selector_handle"]
    display(study_set_bridge["events_index_df"])
    print(f"Loaded Study Set: {study_set_bridge['display_name']} ({study_set_bridge['study_set_id']})")
else:
    print("Set TARGET_STUDY_SET_ID first if you want to load an existing Study Set.")

sel=study_set_selector_handle

NameError: name 'adapter' is not defined

### Signal histogram widget

Presents frequency distributions of any of the session's signal series.

In [5]:
from bodaqs_analysis.widgets.signal_histogram_widget import make_signal_histogram_widget_for_loader
from bodaqs_analysis.widgets.signal_histogram_widget import make_signal_histogram_rebuilder

key_to_ref = sel["get_key_to_ref"]()
events_index_df = sel["get_events_index_df"]()
session_loader = make_session_loader(store=store, key_to_ref=key_to_ref)

hist = make_signal_histogram_rebuilder(sel=sel)
display(hist["out"])


Output()

In [13]:
from bodaqs_analysis.widgets.histogram_core import compute_trimmed_quantile_metrics
import bodaqs_analysis.widgets.histogram_core as hc

m = compute_trimmed_quantile_metrics([1, 2, 3, 4, 5], None)
print(hc.__file__)
print(m)

C:\Users\benco\dev\bodaqs\analysis\bodaqs_analysis\widgets\histogram_core.py
TrimmedQuantileMetrics(n_total=5, n_trim=5, insufficient=False, q25=2.0, q50=3.0, q75=4.0, q90=4.6, q95=4.8, iqr=2.0, skew_q=0.0)


### Event browser

Allows examination of the signal data surrounding any of the session's detected events.

In [10]:
from bodaqs_analysis.widgets.event_browser import make_event_browser_widget_for_loader
from bodaqs_analysis.widgets.event_browser import make_event_browser_rebuilder
from bodaqs_analysis.widgets.loaders import load_all_events_for_selected

key_to_ref = sel["get_key_to_ref"]()
events_index_df = sel["get_events_index_df"]()
session_loader = make_session_loader(store=store, key_to_ref=key_to_ref)

if not key_to_ref:
    raise ValueError("No sessions selected. Select sessions first.")

events_df_sel = load_all_events_for_selected(store, key_to_ref=key_to_ref)

browser = make_event_browser_rebuilder(sel=sel, schema=schema)
display(browser["out"])

Output()

### Metrics scatter plot

Allows any two metrics for a particular event type to be compared as a scatter plot

In [11]:
from bodaqs_analysis.widgets.metric_scatter_widget import make_metric_scatter_widget_for_loader
from bodaqs_analysis.widgets.metric_scatter_widget import make_metric_scatter_rebuilder

key_to_ref = sel["get_key_to_ref"]()
events_index_df = sel["get_events_index_df"]()
session_loader = make_session_loader(store=store, key_to_ref=key_to_ref)

scatter = make_metric_scatter_rebuilder(sel=sel, schema=schema)
display(scatter["out"])

Output()

### Metrics histogram

Presents a frequency distribution of any of an event type's metrics across all detected events

In [12]:
from bodaqs_analysis.widgets.metric_histogram_widget import (
    make_metric_histogram_widget_for_loader,
    make_metric_histogram_rebuilder,
)

#key_to_ref = sel["get_key_to_ref"]()
#events_index_df = sel["get_events_index_df"]()
#session_loader = make_session_loader(store=store, key_to_ref=key_to_ref)

mhist = make_metric_histogram_rebuilder(sel=sel, schema=schema)
display(mhist["out"])

Output()

In [17]:
import plotly.io as pio
from bodaqs_analysis.widgets.session_selector import attach_refresh
pio.renderers.default = "notebook_connected"

refresh = attach_refresh(
    sel,
    rebuild_fns=[browser["rebuild"], hist["rebuild"], scatter["rebuild"], mhist["rebuild"]],
)

# optional manual trigger:
# refresh["trigger"]()
# optional detach:
# refresh["detach"]()
